### Ingestión del archivo "country.json"

In [0]:
%run "../includes/configuration"


In [0]:
%run "../includes/common_functions"

In [0]:
dbutils.widgets.text("p_environment","")
v_environment = dbutils.widgets.get("p_environment")


### Paso 1 - Leer el archivo JSON usando "DataFrameReader" de Spark
#### La documentación para los comandos de Spark la sacamos de https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/index.html

In [0]:
countries_schema = "countryId INT, countryIsoCode STRING, countryName STRING"





In [0]:
#countries_df = spark.read.schema(countries_schema).json("abfss://bronze@moviee.dfs.core.windows.net/country.json")

countries_df = spark.read.schema(countries_schema).json(f"{bronze_folder_path}/country.json")








In [0]:
countries_df.printSchema()

In [0]:
display(countries_df)

### Paso 2 - Eliminar columnas no deseadas del DataFrame
#### Esto sería la otra forma de mantener ciertas columnas que vimos en los primeros 3 notebooks de CSV, en esos vimos como elegir las que queremos, aca vemos como eliminas las que no queremos, esto no tiene nada que ver con que sea json o csv

In [0]:
countries_dropped_df = countries_df.drop("countryIsoCode")




In [0]:
countries_dropped_df = countries_df.drop(countries_df["countryIsoCode"])

In [0]:
from pyspark.sql.functions import col
countries_dropped_df = countries_df.drop(col("countryIsoCode"))

In [0]:
display(countries_dropped_df)

### Paso 3 - Cambiar el nombre de las columnas y añadir "ingestion_date" y "environment"

In [0]:
from pyspark.sql.functions import current_timestamp, lit

#countries_final_df = countries_dropped_df \
#                        .withColumnRenamed("countryId", "country_id") \
#                        .withColumnRenamed("countryName", "country_name") \
#                        .withColumn("ingestion_date", current_timestamp()) \
#                        .withColumn("environment", lit("production"))


countries_final_df = add_ingestion_date(countries_dropped_df) \
                        .withColumnRenamed("countryId", "country_id") \
                        .withColumnRenamed("countryName", "country_name") \
                        .withColumn("environment", lit(v_environment))


In [0]:
display(countries_final_df)

### Paso 4 - Escribir la salida en un formato "Parquet"

In [0]:
#countries_final_df.write.mode("overwrite").parquet("abfss://silver@moviee.dfs.core.windows.net/countries")

countries_final_df.write.mode("overwrite").parquet(f"{silver_folder_path}/countries")

In [0]:
#display(spark.read.parquet("abfss://silver@moviee.dfs.core.windows.net/countries"))

display(spark.read.parquet(f"{silver_folder_path}/countries"))

### Paso 5 - Escribir datos en una managed table en el contenedor silver

#### Esto es un cambio posterior en el notebook, pasaría a reemplazar al paso 4, ya que ahora no quiero más generar archivos de salida en la capa silver a partir del dataframe, sino meter esa info en una tabla administrada por spark, pero dejo igual lo del paso 4, ya que esto se crea dentro de la carpeta _unitystorage y no pisa lo del paso 4

In [0]:
countries_final_df.write.mode("overwrite").format("delta").saveAsTable("movie_silver.countries")

In [0]:
%sql
SELECT * FROM movie_silver.countries

In [0]:
dbutils.notebook.exit("Success")